<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    1.lab.2. Auto MPG Analysis
  </strong>
</div>

### Background

You will be working with the Auto MPG dataset containing fuel consumption information in miles per gallon (MPG) and other attributes for various car models.
The dataset includes:
- 398 instances
- 8 attributes including MPG, cylinders, displacement, horsepower, weight, acceleration, model year and origin

### Description
Your task is to perform an exploratory data analysis using Python. You should use libraries such as `pandas`, `numpy`, and `matplotlib` to:
- Analyze the distribution of features
- Identify correlations between variables
- Explore relationships between features and MPG
- Create meaningful visualizations to support your findings

### Deliverables
- A **ipynb** (Jupyter Notebook) file called **auto-mpg-analysis.ipynb** that contains your code and analysis.
- The **notebook** should be self-contained and include one paragraph summarizing the main findings of your analysis.
- Use the provided **auto-mpg-analysis.ipynb** template as a starting point.

In [ ]:
from seaborn import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Initial configuration

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

### Reading Dataset

Lets read the dataset using pandas.

In [ ]:
df = load_dataset("mpg")

df.head()

### Analyzing the Data

1. **df.shape():** This function help us to understand the number of rows (observations) and columns (features) in the dataset. This gives an overview of the dataset's size and structure.

In [ ]:
print(df.shape)

2. **df.info():** This function help us to understand the dataset by showing the number of records in each column, type of data, whether any values are missing and how much memory the dataset uses.

In [ ]:
df.info()

3. **df.nunique():** This function tells us how many unique values exist in each column which provides insight into the variety of data in each feature.

In [ ]:
df.nunique()

4. **df.describe().T**: This method give me a statistical summary of the DataFrame (Transpose) showing values like count, mean, standard deviation, minimum and quartiles for each numerical column. It helps in summarizing the central tendency and spread of the data.

In [ ]:
df.describe().T

5. **df.columns.tolist():** This converts the column names of the DataFrame into a Python list making it easy to access and manipulate the column names.

In [ ]:
df.columns.tolist()

6. **df.isnull().sum():** This function help us to check the number of missing (null) values in each column of the dataset.

In [ ]:
df.isnull().sum()

7. **df.isna().sum():** This function help us for the same purpose as .isnull(), since both are equivalent and return the count of NaN or undefined values per column.

In [ ]:
df.isna().sum()

Since we find null/undefined/nan values in the horsepower attribute, we have to see which of the following approaches to use for these missing data:

- Delete the rows with missing records.
- Delete the “problematic” column.
- Imputate with the median/mean.
- Imputate using Machine Learning.

Considering that there are only six entries, we can delete them or replace them with the median/mean. However, before deciding whether to use the mean or median, we could see how the values are distributed in this attribute. To do this, we will use a histogram with `sns.histplot(..)`.

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df["horsepower"], bins=20, kde=False)
plt.title("Distribution of Horsepower")
plt.xlabel("Horsepower")
plt.ylabel("Count")
plt.show()

The horsepower distribution is skewed to the right (many cars with low values and few with very high values). This indicates that there are outliers at the top (very large values). Therefore, using the median to fill in the null values is safer than using the mean, because the mean would be affected by the outliers and could be overestimated.

In [ ]:
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

8. **sns.boxplot():** This function help us to visualize the distribution of numerical data and identify possible outliers.

In [ ]:
plt.figure(figsize=(16, 6))
sns.boxplot(data=df)
plt.xticks(rotation=90)
plt.show()

### Observations

- After replacing the missing values (null, NaN, or undefined) in the horsepower column, the dataset no longer contains any missing values, which is great for further analysis.
- The dataset contains a mix of different types of variables:
  - Discrete numerical variables: cylinders, model_year
  - Continuous numerical variables: mpg, displacement, horsepower, weight, acceleration
  - Categorical variables: origin
  - String variable: name
- The dataset contains variables with different ranges:
  - Some variables, like mpg, displacement, weight, and acceleration, have high   ranges, showing significant variation (e.g., fuel efficiency or vehicle weight).
  - Variables such as cylinders and model_year have medium to low ranges, representing more limited variation (e.g., number of cylinders or vehicle age).
  - horsepower also shows a wide range, from low to high power, reflecting different types of vehicles.
- Given the differences in the range of these variables, normalization or standardization will be needed before applying machine learning models.

# Exploratory Data Analysis (EDA)

We will follow these phases:

1. **Target variable analysis:** distribution of the MPG.
2. **Univariate analysis:** distribution of individual features.
3. **Bivariate analysis:** relationship between features and the target variable (MPG).
4. **Correlation matrix and heatmap:** to identify multicollinearity and strong associations.
5. **Joint exploration with PairPlot:** to visualize patterns and separability across multiple features simultaneously.

## 1. Target Variable Analysis

**Explanation:**

- **sns.histplot(...):** Creates a histogram of the mpg variable. 
- **sns.boxplot(..):** Generates a boxplot of the mpg variable. This graph is useful for identifying the median, quartiles, and possible outliers.
- **plt.title(...) and plt.xlabel(...) / plt.ylabel(...):** These commands add descriptive titles and labels to the axes to improve the readability and understanding of each graph.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(df["mpg"], kde=True, bins=30)
plt.title("Distribution of MPG")
plt.xlabel("Miles Per Gallon (MPG)")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.boxplot(y=df["mpg"])
plt.title("Boxplot of MPG")
plt.ylabel("Miles Per Gallon (MPG)")

plt.tight_layout()
plt.show()

## Observations 

- Most vehicles have low fuel consumption, with a peak frequency around 15 MPG.
- The MPG distribution is skewed to the right, meaning that there are fewer vehicles with high fuel efficiency.
- The boxplot confirms these observations. Half of the data (50%) falls within a relatively low MPG range, and the isolated point at the top indicates that there is at least one vehicle with exceptionally high fuel efficiency (outlier).

## 2. Univariate Analysis: Distribution of Characteristics

**Explanation:**

  * **plt.subplot(n_rows, n_cols, i):** Divides the figure into a grid of n rows and n columns, assigning each graph to a position.
  * **sns.histplot(df[characteristic], kde=True, bins=30):** Plots the histogram of each characteristic with a density curve to visualize the distribution.
  * **plt.tight_layout():** Optimizes the spacing between subplots to avoid overlap.

In [ ]:
# Define the grid layout
n_cols = 3
n_rows = 3

plt.figure(figsize=(20, 20))

for i, feature in enumerate(df.columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df[feature], kde=True, bins=30)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

### Observations

- Most variables show a right-skewed distribution, indicating that their values are concentrated at the lower end of the scale. This is evident in variables such as `mpg`, `displacement`, `horsepower`, and `weight`.
- Only the variable acceleration shows a more symmetrical distribution, making it the closest to a normal or Gaussian bell curve distribution.
- Discrete or categorical variables, such as `cylinders`, `model_year`, and `origin`, do not have continuous distributions. Their histograms reflect the frequency of a limited set of possible values.
- The histogram for `name` is not informative due to its high number of unique values.

## 3. Bivariate Analysis: Relationship between Variables and MPG

**Explanation:**

- sns.scatterplot(...):
    - x=df[feature]: Defines the variable on the X-axis.
    - y=df[‘mpg’]: Defines the variable on the Y-axis. 
- These graphs allow us to visually observe whether there is any relationship (linear trend, for example) between characteristics such as horsepower and MPG.

In [ ]:
# Define the grid layout
n_cols = 3
n_rows = 2

# Ignore columns that are not continuous numeric
subset = ["displacement", "horsepower", "weight", "acceleration", "cylinders"]

plt.figure(figsize=(15, 12))

for i, feature in enumerate(subset, 1):
    plt.subplot(n_rows, n_cols, i)
    plt.scatter(df[feature], df["mpg"])
    plt.title(f"MPG vs {feature}")
    plt.xlabel(feature)
    plt.ylabel("MPG")

plt.tight_layout()
plt.show()

### Observations

- Displacement, horsepower, and weight show a clear negative relationship with MPG, as these values increase, MPG tends to decrease.
- The first three (displacement, horsepower, and weight) have a right-skewed distribution, with some outliers.
- Acceleration shows a more uniform and dispersed distribution, without such a marked relationship with MPG.
- In general, outliers are present in several variables, especially displacement and weight, which may require cleaning or transformations for some analyses.
- Lower values of displacement, horsepower, and weight tend to be associated with higher MPG, indicating that lighter cars with less power tend to be more fuel efficient.

## 4. Correlation Matrix and Heatmap

**Explanation:**

- **df.corr():** Calculates the correlation (Pearson coefficient) between the numerical variables in the DataFrame.
- sns.heatmap(...):
  - **cmap=‘coolwarm’:** Uses a palette that differentiates between high (positive) and low (negative) values.
  - **annot=True:** Places the numerical values in each cell.
  - **fmt=".2f":** Defines the numerical format.
  - **linewidths=0.5:** Sets the thickness of the dividing lines.

In [ ]:
matrix_correlation = df.drop(columns=["name", "origin"]).corr()

plt.figure(figsize=(6, 6))
sns.heatmap(matrix_correlation, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix between Variables")
plt.show()

## Observations

- There is a strong negative correlation between fuel consumption and weight, engine displacement, and number of cylinders. We can say that as a car's weight, size, and number of cylinders increase, its more efficiency (fuel consumption) tends to decrease significantly.
- Fuel consumption shows a positive correlation with acceleration (0.42) and model year (0.58). The correlation with model year suggests that newer cars tend to be more fuel efficient.
- There is a very strong positive correlation between the variables cylinders, displacement, horsepower, and weight. 

## 5. Joint Exploration: PairPlot

**Explanation:**

- sns.pairplot():
  - Crea una cuadrícula de gráficos que muestra cada variable contra cada otra variable.
  - diag_kind='kde': En la diagonal se muestran gráficos de densidad (KDE) en lugar de simples histogramas, lo cual ayuda a visualizar la distribución de cada variable.
  - Este gráfico es muy útil para identificar patrones o relaciones de forma simultánea entre múltiples variables.

In [ ]:
subset = ["displacement", "horsepower", "weight", "acceleration", "cylinders", "mpg"]

sns.pairplot(df[subset], diag_kind="kde")
plt.suptitle("Pairplot de Variables Seleccionadas", y=1.02)
plt.show()

### Observations

- The strong positive correlation between displacement, horsepower, weight, and cylinders is confirmed.
- A strong negative correlation between mpg (miles per gallon) and the variables displacement, horsepower, and weight is confirmed.
- Acceleration shows a negative correlation with displacement, horsepower, and weight.
- The diagonal of the pairplot shows the distribution of each variable. It can be seen that most variables, such as displacement, horsepower, and weight, have a right-skewed distribution, while acceleration is more symmetrical.
- The cylinder diagonal shows different peaks, confirming that it is a categorical variable. Looking at the graphs of `cylinders` with other variables, we can see groups of points that correspond to the number of cylinders.

## Conclusions of the Exploratory Analysis

- Most variables have unique distributions. MPG, displacement, horsepower, and weight are skewed to the right, while acceleration is more symmetrical. Categorical variables like cylinders, model year, and origin show specific peaks. This indicates that there are differences in how the data is spread and that some outliers exist.
- Lighter cars with smaller engines and fewer cylinders tend to have higher fuel efficiency. Displacement, horsepower, and weight show a negative relationship with MPG, while acceleration and newer model years are positively related to MPG.
- There are strong correlations among displacement, horsepower, weight, and cylinders. MPG has a strong negative correlation with these variables and a positive correlation with acceleration and model year, this helps to understand which features are related and could be important for predicting fuel efficiency.
- Pairplots and scatter plots confirm the observed trends, showing clear groupings for categorical variables like cylinders and consistent patterns for continuous variables. These visualizations support the relationships and distributions found in the individual analyses.